# A PINN starter for the WNT-style kinetic model

This notebook is the **de-risking step** for building a PINN on the WNT--RA--HOX model.
It does *not* attempt the full 27-state implicit system in one shot. Instead it trains a
PINN on a small, **explicit** 2-variable system (APC `P` and β-catenin `B`) that mirrors
the qualitative structure of your model: a smooth WNT pulse drives β-catenin up, APC is
repressed by β-catenin, and the two settle into an inverse relationship.

It implements the machinery from the two papers you read:

- **SBINN (Yazdani et al. 2020):** input-scaling (`t/T`), per-variable output-scaling,
  data + IC + ODE-residual losses, two-stage training (supervised first, then add residual).
- **Stiff-PINN (Ji et al. 2021):** awareness that stiffness/scale-separation breaks vanilla
  PINNs; here we normalize the residual per variable so small and large states contribute
  comparably to the loss.

The last section explains exactly how to swap in your real system, including the
implicit 4x4 matrix block and the discontinuous WNT signal.


In [ ]:
# If torch is not installed in your environment:
#   pip install torch
import numpy as np
import torch
import torch.nn as nn
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

## Step 1 -- Define the reference system and generate ground truth

We use a clean, explicit nondimensional system as a stand-in for the WNT core:

$$\frac{dP}{dt} = \frac{v_P}{1 + B/K_b} - k_{deg}\,P, \qquad
  \frac{dB}{dt} = v_B\,W(t) - (k_0 + k_c P)\,B.$$

APC `P` is repressed by β-catenin `B`; β-catenin is produced by a smooth WNT pulse `W(t)`
and degraded faster when APC is high. This reproduces the inverse APC/β-catenin behavior
and the damped approach to steady state you saw in the full model, but stays explicit and
non-stiff so the PINN trains reliably. We generate the ground truth with `solve_ivp`.


In [ ]:
# Parameters (nondimensional, chosen for clean O(1) dynamics)
vP, Kb, kdeg = 1.0, 1.0, 0.2
vB, k0, kc   = 2.0, 0.1, 0.5
T = 20.0                      # time horizon
t_on, t_off, ramp = 2.0, 15.0, 0.5   # smooth WNT pulse window

def W_np(t):
    return 0.5 * (np.tanh((t - t_on) / ramp) - np.tanh((t - t_off) / ramp))

def rhs(t, u):
    P, B = u
    dP = vP / (1.0 + B / Kb) - kdeg * P
    dB = vB * W_np(t) - (k0 + kc * P) * B
    return [dP, dB]

u0 = [5.0, 0.0]               # initial conditions [P, B]
t_eval = np.linspace(0.0, T, 1000)
sol = solve_ivp(rhs, (0.0, T), u0, t_eval=t_eval, method="LSODA", rtol=1e-8, atol=1e-10)
ref = sol.y.T                 # shape (1000, 2)

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(sol.t, ref[:, 0], label="APC (P)")
ax[0].plot(sol.t, ref[:, 1], label="beta-catenin (B)")
ax[0].plot(sol.t, W_np(sol.t), "--", color="gray", label="WNT W(t)")
ax[0].set_xlabel("t"); ax[0].legend(); ax[0].set_title("Reference (scipy)")
ax[1].plot(ref[:, 0], ref[:, 1]); ax[1].set_xlabel("APC"); ax[1].set_ylabel("beta-catenin")
ax[1].set_title("Inverse relationship")
plt.tight_layout(); plt.show()

## Step 2 -- Scaling and training data

Following SBINN: the network input is `t/T` (order one), and each output is rescaled by the
magnitude of that variable so the raw network only has to learn O(1) quantities. We also
draw a handful of sparse "observations" from the reference (the supervised anchor) and a
dense set of collocation points where the ODE residual is enforced.


In [ ]:
# Per-variable output scales (mean magnitude of each state)
scales_np = np.maximum(np.mean(np.abs(ref), axis=0), 1e-8)   # shape (2,)
scales = torch.tensor(scales_np, dtype=torch.float32, device=device)
print("output scales:", scales_np)

# Sparse, scattered observations (supervised anchor)
n_obs = 30
obs_idx = np.sort(np.random.choice(len(sol.t), n_obs, replace=False))
t_obs = torch.tensor(sol.t[obs_idx, None], dtype=torch.float32, device=device)
u_obs = torch.tensor(ref[obs_idx], dtype=torch.float32, device=device)

# Collocation points for the ODE residual
n_col = 2000
t_col = torch.tensor(np.linspace(0, T, n_col)[:, None], dtype=torch.float32,
                     device=device, requires_grad=True)

# Initial condition
t0 = torch.zeros(1, 1, device=device)
u0_t = torch.tensor([u0], dtype=torch.float32, device=device)

## Step 3 -- The PINN network

A plain MLP with `tanh` activations, wrapped in input-scaling (`t -> t/T`) and
output-scaling (`raw -> raw * scales`) layers, exactly as in the SBINN architecture.


In [ ]:
class PINN(nn.Module):
    def __init__(self, n_out=2, width=64, depth=4):
        super().__init__()
        layers = [nn.Linear(1, width), nn.Tanh()]
        for _ in range(depth - 1):
            layers += [nn.Linear(width, width), nn.Tanh()]
        layers += [nn.Linear(width, n_out)]
        self.net = nn.Sequential(*layers)

    def forward(self, t):
        t_scaled = t / T                 # input-scaling layer
        raw = self.net(t_scaled)         # O(1) raw outputs
        return raw * scales              # output-scaling layer

model = PINN().to(device)
print(model)

## Step 4 -- ODE residual via automatic differentiation

The network's time-derivative is obtained with `torch.autograd.grad`. The residual is
`du/dt - f(t,u)`, and we **normalize it per variable** (divide by `scales`) so that the
small and large states contribute on equal footing -- the Stiff-PINN lesson about
imbalanced losses across magnitudes.


In [ ]:
def f_torch(t, u):
    P = u[:, 0:1]; B = u[:, 1:2]
    W = 0.5 * (torch.tanh((t - t_on) / ramp) - torch.tanh((t - t_off) / ramp))
    dP = vP / (1.0 + B / Kb) - kdeg * P
    dB = vB * W - (k0 + kc * P) * B
    return torch.cat([dP, dB], dim=1)

def residual(t):
    u = model(t)
    grads = [torch.autograd.grad(u[:, i].sum(), t, create_graph=True)[0]
             for i in range(u.shape[1])]
    dudt = torch.cat(grads, dim=1)
    res = dudt - f_torch(t, u)
    return res / scales                  # per-variable normalization

## Step 5 -- Losses and two-stage training

`L = w_ic * L_ic + w_data * L_data + w_res * L_res`.

Stage 1 trains only the supervised terms (IC + sparse data) so the network quickly lands
near the right curve. Stage 2 adds the ODE residual. This two-stage schedule is the SBINN
recommendation and noticeably stabilizes convergence.


In [ ]:
mse = nn.MSELoss()

def loss_supervised():
    L_ic = mse(model(t0), u0_t)
    L_data = mse(model(t_obs) / scales, u_obs / scales)
    return L_ic, L_data

def loss_residual():
    return (residual(t_col) ** 2).mean()

w_ic, w_data, w_res = 1.0, 1.0, 1.0

# ---- Stage 1: supervised only ----
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
for it in range(1500):
    opt.zero_grad()
    L_ic, L_data = loss_supervised()
    loss = w_ic * L_ic + w_data * L_data
    loss.backward(); opt.step()
    if it % 500 == 0:
        print(f"[stage1 {it:5d}] ic={L_ic.item():.3e} data={L_data.item():.3e}")

# ---- Stage 2: add ODE residual ----
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
for it in range(8000):
    opt.zero_grad()
    L_ic, L_data = loss_supervised()
    L_res = loss_residual()
    loss = w_ic * L_ic + w_data * L_data + w_res * L_res
    loss.backward(); opt.step()
    if it % 1000 == 0:
        print(f"[stage2 {it:5d}] ic={L_ic.item():.3e} "
              f"data={L_data.item():.3e} res={L_res.item():.3e}")

## Step 6 -- Evaluate and compare against the solver


In [ ]:
model.eval()
with torch.no_grad():
    t_test = torch.tensor(sol.t[:, None], dtype=torch.float32, device=device)
    u_pred = model(t_test).cpu().numpy()

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for i, name in enumerate(["APC (P)", "beta-catenin (B)"]):
    ax[i].plot(sol.t, ref[:, i], "b-", label="reference")
    ax[i].plot(sol.t, u_pred[:, i], "r--", label="PINN")
    ax[i].scatter(sol.t[obs_idx], ref[obs_idx, i], s=12, c="k", zorder=3, label="obs")
    ax[i].set_xlabel("t"); ax[i].set_title(name); ax[i].legend()
plt.tight_layout(); plt.show()

rmse = np.sqrt(np.mean((u_pred - ref) ** 2, axis=0))
print("RMSE per variable:", rmse)

## How to extend this to your real WNT--RA--HOX model

This skeleton has every piece you need; scaling up is a matter of replacing parts, in order:

**1. Swap the system.** Replace `f_torch` with the right-hand side of your model and set
`n_out` to the number of states the network predicts. Re-use the per-variable `scales`
computed from a `solve_ivp` reference run (you already have one working).

**2. Handle the implicit 4x4 block.** Four of your derivatives come from `M(u) u_dot = v(u)`,
not from an explicit formula. Do **not** invert `M` inside the network. Instead enforce the
linear relation as an extra residual:
`r_implicit = M(u) @ dudt_block - v(u)`, where `dudt_block` are the autodiff derivatives of
those four outputs. This turns the fragile matrix solve into a clean squared-residual term
and is what sidesteps the overflow issues a numerical solver hits.

**3. Handle the discontinuous WNT signal.** The hard step at tau=3000 and tau=25000 breaks
PINNs. Either smooth it with a `tanh` ramp (as `W_np` already does here), or use domain
decomposition: separate networks on `[0,3000]`, `[3000,25000]`, `[25000,30000]` stitched
with continuity constraints. Start with smoothing.

**4. If training stalls, apply QSSA (Stiff-PINN).** Identify the fast, tiny-magnitude
species (e.g. the destruction-complex intermediates, beta-cat:TCF, near-zero HOX
complexes), set their net rate to zero, solve them algebraically inside `f_torch`, and
exclude them from the network outputs and the loss. This removes the stiffness that a
vanilla PINN cannot handle.

**5. Inverse problem.** To infer the unmeasured HOX parameters, register them as
`nn.Parameter` tensors, add them to the optimizer, and add a data-mismatch loss against
real measurements. The network weights and the unknown rate constants are then learned
simultaneously -- the SBINN inverse setup, which is the genuinely valuable use of a PINN
on this model.

A practical tip from both papers: keep the per-variable residual normalization (Step 4) and
the two-stage schedule (Step 5) as you scale up -- they matter more, not less, once the
state dimension and scale-separation grow.
